# Chapter 05. Cyclical monotonicity and Kantorovich duality

_Source span: printed pp. 51-92; physical PDF pp. 73-114._

This notebook is a standalone computational lesson for the unit named above. The local PDF is used as a map of topics and order, but the explanations, examples, code, and visuals here are rebuilt from scratch. The guiding question is: **Turn optimality into a finite certificate: no improving cycle and matching dual potentials.**

The chapter is treated as a working laboratory rather than a summary. We will translate the mathematical vocabulary into arrays, couplings, graphs, curves, density profiles, and checks that can fail if the idea has been misunderstood. That is especially important in optimal transport, where a word such as "plan", "map", "geodesic", or "curvature" has both a formal definition and a visible computational footprint. The notebook keeps those two readings next to each other.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Optimal-Transport-Old-and-New/part-01-qualitative-description-of-optimal-transport/chapter-05-cyclical-monotonicity-and-kantorovich-duality/05-cyclical-monotonicity-and-kantorovich-duality.ipynb",
  "course_dir": "Optimal-Transport-Old-and-New",
  "course_title": "Optimal Transport: Old and New",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Optimal-Transport-Old-and-New/part-01-qualitative-description-of-optimal-transport/chapter-05-cyclical-monotonicity-and-kantorovich-duality/05-cyclical-monotonicity-and-kantorovich-duality.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Optimal-Transport-Old-and-New/part-01-qualitative-description-of-optimal-transport/chapter-05-cyclical-monotonicity-and-kantorovich-duality/05-cyclical-monotonicity-and-kantorovich-duality.ipynb",
  "notebook_title": "Chapter 05. Cyclical monotonicity and Kantorovich duality",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/ml-geometry.txt",
  "runtime_profile": "ml_geometry"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

BOOK_ROOT = HERE
while BOOK_ROOT.name != "Optimal-Transport-Old-and-New":
    if BOOK_ROOT.parent == BOOK_ROOT:
        raise RuntimeError("Could not locate course root")
    BOOK_ROOT = BOOK_ROOT.parent

for candidate in [BOOK_ROOT, BOOK_ROOT / "scripts"]:
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from scripts.otonn_inventory import UNIT_BY_ID
from utils.artifacts import display_artifact
from utils.transport import cost_matrix, demo_measures, exact_plan, plan_checks, sinkhorn_demo_value
from utils.validation import assert_artifacts, assert_check_flag
from utils.visuals import artifact_paths_for_unit, build_unit_artifacts

UNIT_ID = "chapter-05"
EXPECTED_MODE = "duality"
unit = UNIT_BY_ID[UNIT_ID]
assert unit["mode"] == EXPECTED_MODE


## Translation Guide

This unit's source terms are: `cyclical monotonicity, c-transform, dual potential, slack`. In the notebook they become concrete objects:

- Cyclical monotonicity rules out cheaper cyclic reassignment.
- The c-transform builds potentials adapted to the cost.
- Complementary slackness explains why support points are special.

The computational route is deliberately small. We start with a discrete transport problem because it exposes every marginal, cost entry, and unit of moved mass. We then use the unit's specialized visual mode, `duality`, to focus on the chapter's own geometry. Finally, a sanity cell checks that the generated artifact exists and that the numerical invariant attached to the unit actually passed.

That rhythm is meant to model how to read the book actively. A theorem about existence, a statement about Ricci curvature, or a definition of displacement convexity should leave behind something inspectable: a matrix with the right sums, a path whose mass remains one, a graph whose implications are connected, or a curve whose inequality is numerically visible.


In [ ]:
source_inventory = {
    "unit": unit["id"],
    "label": unit["label"],
    "title": unit["title"],
    "printed_pages": unit["printed"],
    "pdf_pages": unit["pdf"],
    "visual_mode": unit["mode"],
    "terms": unit["terms"],
}
source_inventory


## Concept Lens

1. Cyclical monotonicity rules out cheaper cyclic reassignment. 2. The c-transform builds potentials adapted to the cost. 3. Complementary slackness explains why support points are special.

The common optimal-transport skeleton underneath this unit is the Monge-Kantorovich relaxation. We choose a source measure, a target measure, and a cost matrix. POT solves the finite linear program and returns a plan whose rows and columns must be the prescribed marginals. This modest computation is not a replacement for the theorem; it is a microscope for the theorem's bookkeeping. If the chapter is about duality, the active entries should align with zero slack. If it is about Wasserstein distance, the same matrix becomes a metric computation. If it is about displacement interpolation, the nonzero entries become moving weighted particles. If it is about Ricci curvature, the same interpolation language becomes a place to measure convexity and volume distortion.

The unit-specific layer is: **A matching diagram plus slack matrix for the dual certificate.** This representation was chosen because it makes the chapter's invisible condition visible. The learner should keep asking which object is fixed, which object is optimized, and which invariant is being checked. For this unit, the attached invariant is: **Every active plan entry has near-zero dual slack and every inactive slack is nonnegative.**


In [ ]:
artifact_result = build_unit_artifacts(unit)
paths = artifact_paths_for_unit(unit)
artifact_result, paths


In [ ]:
display_artifact(paths["figure"], width=820)
display_artifact(paths["checks"], width=820)


## Visual Reading

What to inspect: A matching diagram plus slack matrix for the dual certificate. The figure is not meant as decoration; it is a compact diagnostic for the chapter's main move. First locate the conserved quantity or structural relation. In a coupling diagram, that means row and column sums. In a graph, it means reachability of the conceptual route. In curvature and convexity plots, it means the ordering or residual required by the inequality. In a field or heatmap, it means the sign, jump, or determinant that the prose claims should be there.

The adjacent JSON check is part of the lesson. It records the invariant in machine-readable form so that the visual claim is not just a picture. A good habit is to read the figure, predict which Boolean check should be true, then open the JSON and see whether the computation agrees.


In [ ]:
source, target = demo_measures(UNIT_ID)
C = cost_matrix(source.points, target.points, p=2)
plan = exact_plan(source.weights, target.weights, C)
transport_summary = plan_checks(plan, source, target, p=2)
transport_summary


In [ ]:
mode_probe = {
    "mode": EXPECTED_MODE,
    "nonzero_plan_entries": int(np.count_nonzero(plan > 1e-10)),
    "largest_moved_mass": float(plan.max()),
    "source_weight_total": float(source.weights.sum()),
    "target_weight_total": float(target.weights.sum()),
}
if UNIT_ID == "chapter-06":
    mode_probe["geomloss_sinkhorn_probe"] = sinkhorn_demo_value()
mode_probe


## Lab: Make the Theorem Negotiable

Find a three-point permutation that violates cyclical monotonicity after a cost perturbation.

The point of the lab is not to produce a polished theorem. It is to make one hypothesis move while the rest of the setup stays fixed. In optimal transport this is often the quickest way to see why a theorem is phrased with care. A small change in a cost entry can move support from one edge to another. A small change in a density cap can make an interpolation violate a bound. A small change in curvature can reverse the order of a distortion curve.

For this unit, start from the generated data above. Keep the source and target arrays visible, change only one ingredient, and rerun the plan or artifact cell. Record whether the invariant still passes. If it fails, identify whether the failure is a mathematical obstruction, a numerical resolution issue, or simply a sign that the toy model no longer represents the chapter's assumptions.


## Takeaways

- Source span used: printed pp. 51-92; PDF pp. 73-114.
- The chapter vocabulary is rebuilt around `cyclical monotonicity, c-transform, dual potential, slack`.
- The durable artifact lives under `artifacts/chapter-05-cyclical-monotonicity-and-kantorovich-duality` and records both a figure and a JSON invariant check.
- POT supplies the finite Monge-Kantorovich plan used by the common transport lab. GeomLoss is smoke-tested in the course stack and probed in the Wasserstein-distance unit.
- The final sanity cell below checks artifact size, source-map consistency, mass conservation, and the unit-specific invariant.


In [ ]:
final_sanity = {
    "unit_id": UNIT_ID,
    "source_span": "printed pp. 51-92; PDF pp. 73-114",
    "transport_mass_conserved": bool(transport_summary["mass_conserved"]),
    "artifact_paths": {name: str(path.relative_to(BOOK_ROOT)) for name, path in paths.items()},
}
assert unit["printed"] == "51-92"
assert unit["pdf"] == "73-114"
assert transport_summary["mass_conserved"]
assert_artifacts(paths.values())
check_data = assert_check_flag(paths["checks"])
final_sanity["artifact_invariant_ok"] = bool(check_data["invariant_ok"])
final_sanity
